# AML s26 - Principal Component Analysis (PCA) — Example 2


### Load Libraries, ... 

In [21]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
from sklearn.datasets import load_iris
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA as sklearnPCA

np.random.seed(42)
plt.rcParams.update({'figure.dpi': 120, 'font.size': 11, 'axes.spines.top': False, 'axes.spines.right': False})

---
### 1) Load Iris Dataset and perform PCA on it  (4D → 2D)

The Iris dataset has **4 features**.  
We standardize them (zero mean, unit variance) and reduce to 2 PCs for visualization.

In [ ]:
iris       = load_iris()
X_iris     = iris.data
y_iris     = iris.target
feat_names = iris.feature_names
class_names= iris.target_names

# Standardize
scaler    = StandardScaler()
X_scaled  = scaler.fit_transform(X_iris)

# Manual PCA 
C_iris     = np.cov(X_scaled.T)                   # 4×4 covariance matrix
vals, vecs = np.linalg.eigh(C_iris)
order      = np.argsort(vals)[::-1]
vals, vecs = vals[order], vecs[:, order]

Z_iris = X_scaled @ vecs[:, :2]                   # project to 2D

print('Iris covariance matrix (standardized features):')
print(C_iris.round(3))
print()
print('Eigenvalues:', vals.round(4))
print('Variance explained per PC:', (vals/vals.sum()*100).round(1))

### 2) Plot data in a 2D PCA subspace and generate biplot

In [ ]:
colors = ['#e74c3c', '#2ecc71', '#3498db']

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

# 2-D scatter in PC space ─
ax = axes[0]
for cls, col in zip(range(3), colors):
    mask = y_iris == cls
    ax.scatter(Z_iris[mask, 0], Z_iris[mask, 1], label=class_names[cls], color=col, alpha=0.75, edgecolors='white', lw=0.3, s=55)
ax.set_xlabel(f'PC1  ({vals[0]/vals.sum()*100:.1f}% var)')
ax.set_ylabel(f'PC2  ({vals[1]/vals.sum()*100:.1f}% var)')
ax.set_title('Iris — PCA Projection (2D)')
ax.legend()

# Biplot loadings 
ax2 = axes[1]
for cls, col in zip(range(3), colors):
    mask = y_iris == cls
    ax2.scatter(Z_iris[mask,0], Z_iris[mask,1], color=col, alpha=0.25, s=30, edgecolors='none')

scale = 2
short_names = ['sepal len', 'sepal wid', 'petal len', 'petal wid']
for i in range(4):
    ax2.annotate('', xy=(scale*vecs[i,0], scale*vecs[i,1]), xytext=(0,0),  arrowprops=dict(arrowstyle='->', color='black', lw=1.8))
    ax2.text(scale*vecs[i,0]*1.12, scale*vecs[i,1]*1.12,  short_names[i], fontsize=9, ha='center', color='#2c3e50')

ax2.set_xlabel('PC1'); ax2.set_ylabel('PC2')
ax2.set_title('Biplot — Feature Loadings on PCs')
ax2.axhline(0, color='grey', lw=0.7, ls='--')
ax2.axvline(0, color='grey', lw=0.7, ls='--')

plt.tight_layout(); plt.show()

---
### 3)  Generate a Scree Plot — How many components to keep?

Rules of thumb:
* **Elbow rule** — keep PCs up to the "elbow" in the scree plot  
* **Cumulative variance** — keep enough PCs to retain e.g. 95 % of variance

In [ ]:
var_pct  = vals / vals.sum() * 100
cum_var  = np.cumsum(var_pct)
pcs      = np.arange(1, len(vals)+1)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# Scree plot
ax = axes[0]
ax.plot(pcs, var_pct, 'o-', color='steelblue', lw=2, ms=8)
ax.set_xlabel('Principal Component'); ax.set_ylabel('% Variance Explained')
ax.set_title('Scree Plot')
ax.set_xticks(pcs)
for x, y in zip(pcs, var_pct):
    ax.text(x, y+1, f'{y:.1f}%', ha='center', fontsize=9, color='steelblue')

# Cumulative variance
ax2 = axes[1]
ax2.bar(pcs, cum_var, color='darkorchid', alpha=0.7, edgecolor='white')
ax2.axhline(95, color='crimson', ls='--', lw=1.5, label='95% threshold')
ax2.set_xlabel('Number of PCs kept'); ax2.set_ylabel('Cumulative % Variance')
ax2.set_title('Cumulative Explained Variance')
ax2.set_xticks(pcs); ax2.set_ylim(0, 105)
ax2.legend()
for x, y in zip(pcs, cum_var):
    ax2.text(x, y+1.5, f'{y:.1f}%', ha='center', fontsize=9)

plt.tight_layout(); plt.show()

needed = np.searchsorted(cum_var, 95) + 1
print(f'PCs needed to explain ≥ 95% variance: {needed}')

---
### 4)  Validation against scikit-learn

Confirm our manual PCA matches `sklearn.decomposition.PCA`.

In [ ]:
sk_pca   = sklearnPCA(n_components=2)
Z_sk     = sk_pca.fit_transform(X_scaled)

# Signs may differ — compare absolute scores
match = np.allclose(np.abs(Z_iris), np.abs(Z_sk), atol=1e-6)
print('Manual PCA matches sklearn:', match)
print()
print('sklearn explained variance ratio:', sk_pca.explained_variance_ratio_.round(4))
print('Manual   explained variance ratio:', (vals[:2]/vals.sum()).round(4))